# 第6章　結果を確かめる ― 評価指標の基礎

**『医療診断支援AI開発　入門編 ― ゼロから動かす（入門編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-intro

## 6.3　閾値とROC曲線・PR曲線

In [ ]:
from sklearn.metrics import recall_score, precision_score, roc_auc_score

recall = recall_score(y_true, y_pred)        # 感度。既定では「1」を陽性として扱う点に注意
precision = precision_score(y_true, y_pred)  # 適合率（警告の確からしさ）
auc = roc_auc_score(y_true, y_prob)          # AUC（総合力、y_probは確率）
print(f"感度={recall:.2f} 適合率={precision:.2f} AUC={auc:.3f}")

## 数字で追う ― 同じAIでも、有病率が変われば「陽性的中率」は激変する

In [ ]:
def ppv(sens, spec, prev):
    tp = sens * prev                 # 真陽性の割合
    fp = (1 - spec) * (1 - prev)     # 偽陽性の割合
    return tp / (tp + fp)

for prev in [0.01, 0.05, 0.10, 0.50]:
    print(f"有病率 {prev:>4.0%} → 陽性的中率 = {ppv(0.9, 0.9, prev):.1%}")
# 有病率   1% → 陽性的中率 =  8.3%
# 有病率   5% → 陽性的中率 = 32.1%
# 有病率  10% → 陽性的中率 = 50.0%
# 有病率  50% → 陽性的中率 = 90.0%

## 数字で追う ― 1枚の混同行列から、6つの指標を電卓で出す

In [ ]:
tp, fn, fp, tn = 36, 4, 24, 136
acc  = (tp + tn) / (tp + fn + fp + tn)
sens = tp / (tp + fn)
spec = tn / (tn + fp)
ppv  = tp / (tp + fp)
npv  = tn / (tn + fn)
f1   = 2 * ppv * sens / (ppv + sens)
print(f"正解率{acc:.2f} 感度{sens:.2f} 特異度{spec:.2f} PPV{ppv:.2f} NPV{npv:.2f} F1{f1:.2f}")
# 正解率0.86 感度0.90 特異度0.85 PPV0.60 NPV0.97 F10.72

## 手を動かす ― ROC曲線を描き、閾値表と混同行列で「運用点」を決める

In [ ]:
import matplotlib.pyplot as plt
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix

# 合成データ：陰性120例・陽性40例（有病率25%）。
# 陽性のほうが「病気らしさ」がやや高いが、分布は重なっている＝現実的な難しさ
rng = np.random.default_rng(0)
y_true = np.r_[np.zeros(120), np.ones(40)].astype(int)
y_prob = np.r_[rng.normal(0.35, 0.15, 120),
               rng.normal(0.60, 0.15,  40)].clip(0, 1)

fpr, tpr, thr = roc_curve(y_true, y_prob)   # 各閾値での 偽陽性率・真陽性率
auc = roc_auc_score(y_true, y_prob)

plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, lw=2, label=f"AUC = {auc:.3f}")
plt.plot([0, 1], [0, 1], "--", color="gray", label="chance")   # 偶然の対角線
plt.xlabel("1 - Specificity (FPR)"); plt.ylabel("Sensitivity (TPR)")
plt.title("ROC curve"); plt.legend(); plt.show()

In [ ]:
print(f"{'閾値':>6} {'感度':>6} {'特異度':>7} {'PPV':>6}")
for t in [0.30, 0.40, 0.50, 0.60, 0.70]:
    pred = (y_prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn)
    spec = tn / (tn + fp)
    ppv  = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    print(f"{t:5.2f} {sens:6.2f} {spec:7.2f} {ppv:6.2f}")

In [ ]:
target = 0.95
ok = tpr >= target
if ok.any():
    t_op = thr[ok][0]           # 感度0.95以上を満たす中で最も高い閾値＝特異度を最大化
    i    = np.where(ok)[0][0]
    print(f"運用閾値 = {t_op:.2f}（感度 {tpr[i]:.2f}, 特異度 {1 - fpr[i]:.2f}）")
else:
    print("このモデルでは、どの閾値でも感度0.95に届きません")

In [ ]:
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, (y_prob >= t_op).astype(int), labels=[0, 1])
plt.figure(figsize=(4, 4))
plt.imshow(cm, cmap="Blues")
for r in range(2):
    for c in range(2):
        plt.text(c, r, cm[r, c], ha="center", va="center",
                 color="white" if cm[r, c] > cm.max() / 2 else "black")
plt.xticks([0, 1], ["pred:neg", "pred:pos"])
plt.yticks([0, 1], ["true:neg", "true:pos"])
plt.xlabel("predicted"); plt.ylabel("actual")
plt.title(f"Confusion matrix (thr={t_op:.2f})"); plt.colorbar(); plt.show()